In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RoPE(nn.Module):
    def __init__(self, dim:int, max_seq_len: int, theta: float = 10000.0):
        """
        初始化 RoPE 模块。
        Args:
            dim (int): 输入 embeddings 的最后一个维度的大小 (通常是 head_dim)。
                       注意：dim 必须是偶数，因为 RoPE 成对操作维度。
            max_seq_len (int): 模型能处理的最大序列长度。
            theta (float): RoPE 中的 base value，用于计算频率。
        """
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim 必须是偶数，因为 RoPE 成对操作维度。")

        self.dim = dim
        self.max_seq_len = max_seq_len
        self.theta = theta

        # freqs shape: (dim / 2)
        # freqs_i = 1 / (theta ^ (2i/dim) )
        # freqs: [freqs_0, freqs_1, ..., freqs_{dim/2 - 1}]
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))

        # 位置t
        # t shape: (max_seq_len)
        # t: [0, 1, ..., max_seq_len - 1]
        t = torch.arange(max_seq_len, device = freqs.device)

        # cal m * freqs_i
        # freqs_outer shape: (max_seq_len, dim / 2)
        # freqs_outer[m, i] = m * freqs_i
        freqs_outer = torch.outer(t, freqs)

        # trans m * freqs_i to 复数形式: cos(m * theta_i) + j * sin(m * theta_i)
        # freqs_cis shape: (max_seq_len, dim / 2)
        # 每个元素是一个复数 e^(j * m * freqs_i)
        freqs_cis = torch.polar(torch.ones_like(freqs_outer), freqs_outer)

        # 注册为 buffer，这样它会被移动到正确的设备并且不会被视为模型参数
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, x: torch.Tensor, current_pos: int = 0):
        """
        对输入张量 x 应用 RoPE。
        Args:
            x (torch.Tensor): 输入张量，形状通常是 (batch_size, num_heads, seq_len, head_dim)
                              或者 (batch_size, seq_len, num_heads, head_dim)
                              或者任何最后一个维度是 self.dim 的张量。
                              这里我们假设最后一个维度是 head_dim。
            current_pos (int): 当前生成token的位置（用于KV Cache的场景）。
                               如果处理整个序列，一般为0。
        Returns:
            torch.Tensor: 应用了 RoPE 的张量，形状与 x 相同。
        """

        # 取当前序列对应的旋转因子
        seq_len = x.shape[-2] # 假设倒数第二个维度是序列长度
        # self.freqs_cis形状：(max_seq_len, dim//2)
        freqs_cis_to_apply = self.freqs_cis[current_pos : current_pos + seq_len, :]

        # 把 x 变成复数
        # x shape: (B, H, S, D)
        # x_complex shape: (B, H, S, D/2)
        x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))


        # 对齐形状，准备广播
        # shape: (S, D/2) -> (1, 1, S, D/2)
        freqs_cis_reshaped = freqs_cis_to_apply.unsqueeze(0).unsqueeze(0)

        # 复数乘法 = 旋转
        x_rotated = x_complex * freqs_cis_reshaped

        # 变回实数
        # 复数 → 实部和虚部拆开：[..., real, imag]
        # 形状： (B, H, S, D/2, 2)
        x_out = torch.view_as_real(x_rotated)

        # reshape 回原形状
        # 从 (..., D/2, 2)→ (..., D)
        x_out = x_out.reshape(*x.shape)

        return x_out.type_as(x)



In [2]:
def example_rope():
    print("--- RoPE Example ---")
    batch_size = 1
    num_heads = 2
    seq_len = 4
    head_dim = 6
    max_seq_len_rope = 10

    # 创建 RoPE 实例
    rope = RoPE(dim = head_dim, max_seq_len = max_seq_len_rope)

    # 模拟一个 Q 或 K 张量 (batch_size, num_heads, seq_len, head_dim)
    # 或者 (batch_size, seq_len, num_heads, head_dim)
    # RoPE作用于最后一个维度，所以我们传入 (B, H, S, D_h)
    x = torch.randn(batch_size, num_heads, seq_len, head_dim)
    print(f"Original x (first head, first token): {x[0, 0, 0, :]}")

    # 应用 RoPE
    # 假设这是序列的开始，current_pos=0
    x_rotated = rope(x, current_pos = 0)
    print(f"Rotated x (first head, first token, pos 0): {x_rotated[0, 0, 0, :]}")

    # 检查不同位置的旋转是否不同
    # 假设我们只取 x 的最后一个 token，并告诉 RoPE 它的位置是 seq_len - 1
    # 这模拟了 KV cache 中，只对新 token 应用 RoPE 的情况
    last_token_original = x[:, :, -1:, :] # (B, H, 1, D_h)
    last_token_rotated = rope(last_token_original, current_pos = seq_len - 1)
    print(f"Rotated x (last token, pos {seq_len-1}): {last_token_rotated[0, 0, 0, :]}")
    print(f"Directly from full rotated (last token): {x_rotated[0, 0, -1, :]}")
    assert torch.allclose(last_token_rotated, x_rotated[:,:,-1:,:], atol=1e-6), "RoPE on slice failed"


    # 比较第一个和第二个 token 的旋转结果
    print(f"Rotated x (first head, second token, pos 1): {x_rotated[0, 0, 1, :]}")
    # 它们应该不同，因为位置不同
    assert not torch.allclose(x_rotated[0,0,0,:], x_rotated[0,0,1,:]), "Tokens at different positions should have different rotations"

    # 验证 RoPE 的属性：保持范数 (近似)
    # 对于 (x_1, x_2), 旋转后 (x_1', x_2')
    # x_1^2 + x_2^2 应约等于 x_1'^2 + x_2'^2
    x_pair_original = x[0,0,0,0:2]
    x_pair_rotated = x_rotated[0,0,0,0:2]
    norm_orig = torch.sum(x_pair_original**2)
    norm_rot = torch.sum(x_pair_rotated**2)
    print(f"Norm of first pair original: {norm_orig:.4f}, rotated: {norm_rot:.4f}")
    assert torch.isclose(norm_orig, norm_rot, atol=1e-5), "RoPE should preserve norm of pairs"
    print("RoPE example finished.\n")



example_rope()

--- RoPE Example ---
Original x (first head, first token): tensor([-0.6266,  1.5063,  1.0343,  2.2049, -1.5802,  0.8844])
Rotated x (first head, first token, pos 0): tensor([-0.6266,  1.5063,  1.0343,  2.2049, -1.5802,  0.8844])
Rotated x (last token, pos 3): tensor([ 0.4940,  1.0280, -0.7645,  0.5755, -1.0870, -0.7454])
Directly from full rotated (last token): tensor([ 0.4940,  1.0280, -0.7645,  0.5755, -1.0870, -0.7454])
Rotated x (first head, second token, pos 1): tensor([-1.3745, -0.5579, -0.6654, -0.4525,  0.3083, -0.6592])
Norm of first pair original: 2.6616, rotated: 2.6616
RoPE example finished.

